# Compare HP k15-k20 vs TEA for Family, Superfamily, and Fold

This notebook compares HP (hydrophobic-polar) encoding across different ksizes (k=15 to k=20) against TEA baseline.

## TEA Paper Definition:

From the TEA paper:
> The family and superfamily benchmark plots measure the sensitivity of detected TPs 
> (same family, and same superfamily but not same family respectively) up to the first FP 
> (where FP is defined as hit from a different fold).

So:
- **Family**: TP = same family, FP = different fold
- **Superfamily**: TP = same superfamily BUT NOT same family, FP = different fold
- **Fold**: TP = same fold, FP = different fold

In [2]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

sns.set_style('whitegrid')
%matplotlib inline

## Helper Functions

In [3]:
def extract_scope_levels(name):
    """Extract SCOPe hierarchical levels from protein name."""
    parts = name.split()
    if len(parts) < 2:
        return None
    
    lineage = parts[1]
    lineage_parts = lineage.split('.')
    
    if len(lineage_parts) < 4:
        return None
    
    # SCOPe format: class.fold.superfamily.family
    cls = lineage_parts[0]
    fold = f"{cls}.{lineage_parts[1]}"
    superfamily = f"{fold}.{lineage_parts[2]}"
    family = f"{superfamily}.{lineage_parts[3]}"
    
    return {
        'class': cls,
        'fold': fold,
        'superfamily': superfamily,
        'family': family
    }

def add_scope_levels(df):
    """Add SCOPe hierarchical levels and match indicators following TEA paper definition."""
    # Extract levels
    query_levels = df['query_name'].map_elements(extract_scope_levels, return_dtype=pl.Struct).alias('query_levels')
    target_levels = df['target_name'].map_elements(extract_scope_levels, return_dtype=pl.Struct).alias('target_levels')
    
    # Add level columns
    df = df.with_columns([
        query_levels.struct.field('family').alias('query_family'),
        query_levels.struct.field('superfamily').alias('query_superfamily'),
        query_levels.struct.field('fold').alias('query_fold'),
        query_levels.struct.field('class').alias('query_class'),
        target_levels.struct.field('family').alias('target_family'),
        target_levels.struct.field('superfamily').alias('target_superfamily'),
        target_levels.struct.field('fold').alias('target_fold'),
        target_levels.struct.field('class').alias('target_class'),
    ])
    
    # Add match indicators following TEA paper definition
    df = df.with_columns([
        # Family: TP = same family, FP = different fold
        (pl.col('query_family') == pl.col('target_family')).alias('family_tp'),
        (pl.col('query_fold') != pl.col('target_fold')).alias('family_fp'),
        
        # Superfamily: TP = same superfamily BUT NOT same family, FP = different fold
        ((pl.col('query_superfamily') == pl.col('target_superfamily')) & 
         (pl.col('query_family') != pl.col('target_family'))).alias('superfamily_tp'),
        (pl.col('query_fold') != pl.col('target_fold')).alias('superfamily_fp'),
        
        # Fold: TP = same fold, FP = different fold (this means no FPs by definition)
        (pl.col('query_fold') == pl.col('target_fold')).alias('fold_tp'),
        (pl.col('query_fold') != pl.col('target_fold')).alias('fold_fp'),
    ])
    
    return df

def calculate_sensitivity_tea_definition(df, metric_col, tp_col, fp_col, group_by_ksize=True):
    """Calculate sensitivity to first FP following TEA paper definition."""
    # Remove self-hits
    if 'query_md5' in df.columns and 'target_md5' in df.columns:
        df = df.filter(pl.col('query_md5') != pl.col('target_md5'))
    
    group_cols = ['query_name']
    if group_by_ksize:
        group_cols = ['ksize'] + group_cols
    
    queries = df.select(group_cols).unique()
    results = []
    
    for row in queries.iter_rows():
        if group_by_ksize:
            ksize, query_name = row
            query_df = df.filter((pl.col('ksize') == ksize) & (pl.col('query_name') == query_name))
        else:
            query_name = row[0]
            query_df = df.filter(pl.col('query_name') == query_name)
            ksize = None
        
        # Sort by metric (descending = better)
        query_df = query_df.sort(metric_col, descending=True)
        
        # Get TP and FP indicators
        tps = query_df[tp_col].to_list()
        fps = query_df[fp_col].to_list()
        
        # Count TPs
        n_positives = sum(tps)
        if n_positives == 0:
            continue
        
        # Find first FP
        first_fp = next((i for i, is_fp in enumerate(fps) if is_fp), None)
        
        if first_fp is None:
            sensitivity = 1.0
        elif first_fp == 0:
            sensitivity = 0.0
        else:
            # Count TPs before first FP
            tps_before_fp = sum(tps[:first_fp])
            sensitivity = min(tps_before_fp / n_positives, 1.0)
        
        result = {'query_name': query_name, 'sensitivity_to_first_fp': sensitivity}
        if group_by_ksize:
            result['ksize'] = ksize
        results.append(result)
    
    return pl.DataFrame(results)

## Load HP Results

In [4]:
hp_results_dir = Path('/Users/olga/data/scope/results-2025-12-25-average_kmer_rarity')
hp_files = sorted(hp_results_dir.glob('*hp*.csv'))

print(f"Found {len(hp_files)} HP CSV files")
for f in hp_files:
    print(f"  {f.name} ({f.stat().st_size / (1024**3):.2f} GB)")

Found 6 HP CSV files
  astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k15.scaled1.kmerseek.results.csv (57.88 GB)
  astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k16.scaled1.kmerseek.results.csv (28.87 GB)
  astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k17.scaled1.kmerseek.results.csv (14.43 GB)
  astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k18.scaled1.kmerseek.results.csv (7.23 GB)
  astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k19.scaled1.kmerseek.results.csv (3.62 GB)
  astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k20.scaled1.kmerseek.results.csv (1.82 GB)


In [ ]:
# Load HP data with size-aware reading
MAX_ROWS = 15_000_000
SIZE_THRESHOLD_GB = 8

hp_dfs = []
for hp_file in hp_files:
    file_size_gb = hp_file.stat().st_size / (1024**3)
    ksize = int(hp_file.stem.split('.k')[1].split('.')[0])
    
    print(f"Loading k={ksize} ({file_size_gb:.2f} GB)...", end=" ")
    
    if file_size_gb > SIZE_THRESHOLD_GB:
        df = pl.read_csv(hp_file, n_rows=MAX_ROWS)
        print(f"read {MAX_ROWS:,} rows")
    else:
        df = pl.read_csv(hp_file)
        print(f"read {df.shape[0]:,} rows")
    
    df = df.with_columns(pl.lit(ksize).alias('ksize'))
    hp_dfs.append(df)

hp_data = pl.concat(hp_dfs)
print(f"\nTotal HP data: {hp_data.shape}")
print(f"Ksizes: {sorted(hp_data['ksize'].unique().to_list())}")

Loading k=15 (57.88 GB)... 

In [ ]:
# Add SCOPe levels
print("Adding SCOPe hierarchical levels to HP data...")
hp_data = add_scope_levels(hp_data)
print(f"HP data shape: {hp_data.shape}")
hp_data.head()

Adding SCOPe hierarchical levels to HP data...


## Load TEA Results

In [ ]:
tea_dir = Path('/Users/olga/code/2024-kmerseek-analysis/data/tea_scope40_rocx_files')
tea_file = tea_dir / 'tea_all.rocx'

if tea_file.exists():
    print(f"Loading TEA data from {tea_file.name}...")
    tea_data = pl.read_csv(tea_file)
    print(f"TEA data shape: {tea_data.shape}")
    
    # Add SCOPe levels
    print("Adding SCOPe hierarchical levels to TEA data...")
    tea_data = add_scope_levels(tea_data)
    print(f"TEA data shape after adding levels: {tea_data.shape}")
    tea_data.head()
else:
    print(f"TEA file not found: {tea_file}")
    tea_data = None

NameError: name 'Path' is not defined

## Calculate Sensitivity for All Metrics and Levels

In [ ]:
# Define metrics and levels
metrics = {
    'TF-IDF': 'tfidf',
    'Max Containment': 'max_containment',
    'Jaccard': 'jaccard'
}

levels_config = {
    'Family': ('family_tp', 'family_fp'),
    'Superfamily': ('superfamily_tp', 'superfamily_fp'),
    'Fold': ('fold_tp', 'fold_fp')
}

# Set up colors
ksizes = sorted(hp_data['ksize'].unique().to_list())
ksize_colors = dict(zip(ksizes, sns.color_palette('viridis', len(ksizes))))

## Plot: All Levels with TF-IDF (Default Metric)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
metric_col = 'tfidf'

for ax, (level_name, (tp_col, fp_col)) in zip(axes, levels_config.items()):
    print(f"Plotting {level_name}...")
    
    # Plot HP for each ksize
    for ksize in ksizes:
        ksize_data = hp_data.filter(pl.col('ksize') == ksize)
        sens_df = calculate_sensitivity_tea_definition(ksize_data, metric_col, tp_col, fp_col, group_by_ksize=False)
        sens_df = sens_df.sort('sensitivity_to_first_fp')
        
        n_queries = sens_df.shape[0]
        plot_df = sens_df.with_columns(
            ((pl.arange(0, n_queries) + 1) / n_queries).alias('fraction_queries')
        ).to_pandas()
        
        ax.plot(plot_df['fraction_queries'], plot_df['sensitivity_to_first_fp'],
                label=f'HP k={ksize}', color=ksize_colors[ksize], linewidth=2)
    
    # Plot TEA
    if tea_data is not None:
        tea_sens_df = calculate_sensitivity_tea_definition(tea_data, metric_col, tp_col, fp_col, group_by_ksize=False)
        tea_sens_df = tea_sens_df.sort('sensitivity_to_first_fp')
        
        n_tea_queries = tea_sens_df.shape[0]
        tea_plot_df = tea_sens_df.with_columns(
            ((pl.arange(0, n_tea_queries) + 1) / n_tea_queries).alias('fraction_queries')
        ).to_pandas()
        
        ax.plot(tea_plot_df['fraction_queries'], tea_plot_df['sensitivity_to_first_fp'],
                label='TEA', color='purple', linewidth=2.5, linestyle='--')
    
    ax.set_xlabel('Fraction of queries', fontsize=12)
    ax.set_ylabel('Sensitivity up to 1st FP', fontsize=12)
    ax.set_title(level_name, fontsize=14, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hp_k15_k20_vs_tea_tfidf.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Plot: Separate Figure for Each Metric

In [ ]:
for metric_name, metric_col in metrics.items():
    print(f"\n{'='*80}")
    print(f"Generating plots for {metric_name}")
    print('='*80)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    for ax, (level_name, (tp_col, fp_col)) in zip(axes, levels_config.items()):
        print(f"  {level_name}...", end=" ")
        
        # Plot HP
        for ksize in ksizes:
            ksize_data = hp_data.filter(pl.col('ksize') == ksize)
            sens_df = calculate_sensitivity_tea_definition(ksize_data, metric_col, tp_col, fp_col, group_by_ksize=False)
            sens_df = sens_df.sort('sensitivity_to_first_fp')
            
            n = sens_df.shape[0]
            plot_df = sens_df.with_columns(((pl.arange(0, n) + 1) / n).alias('frac')).to_pandas()
            ax.plot(plot_df['frac'], plot_df['sensitivity_to_first_fp'],
                    label=f'HP k={ksize}', color=ksize_colors[ksize], linewidth=2)
        
        # Plot TEA
        if tea_data is not None:
            tea_sens = calculate_sensitivity_tea_definition(tea_data, metric_col, tp_col, fp_col, group_by_ksize=False)
            tea_sens = tea_sens.sort('sensitivity_to_first_fp')
            n_tea = tea_sens.shape[0]
            tea_plot = tea_sens.with_columns(((pl.arange(0, n_tea) + 1) / n_tea).alias('frac')).to_pandas()
            ax.plot(tea_plot['frac'], tea_plot['sensitivity_to_first_fp'],
                    label='TEA', color='purple', linewidth=2.5, linestyle='--')
        
        ax.set_xlabel('Fraction of queries', fontsize=12)
        ax.set_ylabel('Sensitivity up to 1st FP', fontsize=12)
        ax.set_title(level_name, fontsize=14, fontweight='bold')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        
        print("Done")
    
    fig.suptitle(f'{metric_name}: HP k15-k20 vs TEA', fontsize=16, y=1.02)
    plt.tight_layout()
    
    filename = f"hp_k15_k20_vs_tea_{metric_name.lower().replace(' ', '_').replace('-', '_')}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")